# IsoCourt — CNN+LSTM (ResNet50) Training on FineBadminton-20K

Registry category: **`cnn_lstm`** | Script: `train_full.py` | Checkpoint: `badminton_model.pth`

**Runtime → Change runtime type → GPU (A100 recommended, T4 works)**

In [ ]:
import torch, subprocess, os
assert torch.cuda.is_available(), 'No GPU — change runtime to GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1 — Mount Google Drive (checkpoints persist across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/IsoCourt/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)
print('Checkpoints will be saved to:', DRIVE_CKPT)

## 2 — Clone repo & install dependencies

In [ ]:
REPO = 'https://github.com/Navneethd8/IsoCourt.git'
BRANCH = 'main'

if not os.path.isdir('/content/IsoCourt'):
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 -b {BRANCH} {REPO} /content/IsoCourt
else:
    !cd /content/IsoCourt && git pull --ff-only

%cd /content/IsoCourt

In [ ]:
!pip install -q opencv-python-headless mediapipe mlflow tqdm timm transformers safetensors huggingface_hub

## 3 — Download FineBadminton-20K & prepare data

In [ ]:
DATA_DIR = '/content/IsoCourt/backend/data/FineBadminton-20K'
MERGED_JSON = '/content/IsoCourt/backend/data/transformed_combined_rounds_output_en_evals_translated.json'

if not os.path.isfile(MERGED_JSON):
    print('Downloading FineBadminton-20K from HuggingFace Hub...')
    !python backend/pipelines/vlm/common/prepare_finebadminton_20k.py \
        --local-dir {DATA_DIR} --max-workers 8
else:
    print(f'Merged JSON already exists: {MERGED_JSON}')

## 4 — Train CNN+LSTM (ResNet50 + LSTM)

In [ ]:
EPOCHS = 60
SAVE_PATH = os.path.join(DRIVE_CKPT, 'badminton_model.pth')
POSE_CACHE = os.path.join(DRIVE_CKPT, 'pose_cache_staeformer.pt')
USE_POSE = True

import sys
sys.path.insert(0, '/content/IsoCourt/backend')

from pipelines.training.train_full import train_full

DATA_ROOT = '/content/IsoCourt/backend/data'
LIST_FILE = MERGED_JSON

train_full(
    data_root=DATA_ROOT,
    list_file=LIST_FILE,
    epochs=EPOCHS,
    batch_size=4,
    lr=1e-4,
    device='cuda',
    hidden_size=128,
    save_path=SAVE_PATH,
    pose_cache_path=POSE_CACHE,
    use_pose=USE_POSE,
)

In [ ]:
print('Training complete.')
print(f'Best checkpoint: {SAVE_PATH}')
print(f'Pose cache: {POSE_CACHE}')
!ls -lh {DRIVE_CKPT}/*.pth 2>/dev/null || echo 'No checkpoints found'